In [21]:
import json
import gradio as gr
from openai import OpenAI
from IPython.display import display, Markdown
from dotenv import load_dotenv

In [22]:
def record_email(email: str):
    with open("user_emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")

In [11]:
record_email("test@example.com")

In [23]:
tools = [{
    "type": "function",
    "function": {
        "name": "record_email",
        "description": "Records an email to the file.",
        "parameters": {
            "type": "object",
            "properties": {
                "email": {
                    "type": "string",
                    "description": "The email address to be recorded into the file."
                }
            },
            "required": ["email"]
        }
    }
}]

In [24]:
system_prompt = "You are an assistant that records user emails into a file."

In [19]:
def chat_with_user(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model="gpt-5.4-nano", messages=messages, tools=tools)
    assistant_message = response.choices[0].message
    
    messages.append(assistant_message)
    
    if assistant_message.tool_calls:
        for tool_call in assistant_message.tool_calls:
            fn_name = tool_call.function.name
            fn_args = tool_call.function.arguments
            if fn_name == "record_email":
                result = record_email(fn_args["email"])
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })
        final_response = client.chat.completions.create(model="gpt-5.4-nano", messages=messages, tools=tools)
        return final_response.choices[0].message.content
    return assistant_message.content

In [28]:
gr.ChatInterface(fn=chat_with_user).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/tkarmysh/tlnt-playground/record-email-agent/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 870, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/tkarmysh/tlnt-playground/record-email-agent/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 418, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/tkarmysh/tlnt-playground/record-email-agent/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/tkarmysh/tlnt-playground/record-email-agent/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1961, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/tkarmysh/tlnt